# Candidate slowdown-label definition

This notebook compares present-time diagnostic label rules based only on sustained resource pressure. It does not create future targets, train models, or claim that any rule is final.

### 1. Define paths and fingerprint protected inputs

**What the cell does:** Imports analysis libraries, defines repository-relative paths, and records SHA-256 checksums for every input plus the protected cleaned dataset.  
**Why it is important:** Label analysis must be reproducible and non-destructive.  
**What to understand:** The printed fingerprints identify the exact files used and will be compared again at the end.

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SEGMENTED_PATH = PROJECT_ROOT / "data" / "interim" / "segmented_metrics.csv"
SEGMENT_SUMMARY_PATH = PROJECT_ROOT / "reports" / "segment_summary.csv"
GAP_VALIDATION_PATH = PROJECT_ROOT / "reports" / "gap_validation.csv"
CLEANED_PROTECTED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_metrics.csv"
RULE_COMPARISON_PATH = PROJECT_ROOT / "reports" / "label_rule_comparison.csv"
MACHINE_DISTRIBUTION_PATH = PROJECT_ROOT / "reports" / "label_distribution_by_machine.csv"
RUN_DISTRIBUTION_PATH = PROJECT_ROOT / "reports" / "label_distribution_by_run.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

protected_paths = [SEGMENTED_PATH, SEGMENT_SUMMARY_PATH, GAP_VALIDATION_PATH, CLEANED_PROTECTED_PATH]
missing_paths = [str(path) for path in protected_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Required files are missing: {missing_paths}")

hashes_before = {path: sha256_file(path) for path in protected_paths}
for path, fingerprint in hashes_before.items():
    print(f"{path}: {fingerprint}")

### 2. Load inputs and verify sequence identity

**What the cell does:** Loads copies of the segmented metrics and supporting reports, verifies machine/run/segment/timestamp fields, and safely parses timestamps.  
**Why it is important:** Rolling calculations must remain inside valid chronological sequences and must never bridge a gap boundary.  
**What to understand:** Every metrics row has the keys needed for bounded rolling windows, and segment IDs agree with the segment report.

In [ ]:
segmented_input = pd.read_csv(SEGMENTED_PATH)
segment_summary_input = pd.read_csv(SEGMENT_SUMMARY_PATH)
gap_validation_input = pd.read_csv(GAP_VALIDATION_PATH)

required_identity = {"machine_id", "run_id", "segment_id", "timestamp"}
resource_columns = {"cpu_pct", "ram_pct", "swap_pct", "disk_latency_ms", "context_switches_per_s"}
missing_columns = (required_identity | resource_columns) - set(segmented_input.columns)
if missing_columns:
    raise KeyError(f"Required segmented-metric columns are missing: {sorted(missing_columns)}")

label_analysis = segmented_input.copy(deep=True)
label_analysis["_source_row"] = np.arange(len(label_analysis))
label_analysis["_timestamp_dt"] = pd.to_datetime(label_analysis["timestamp"], errors="coerce", utc=True)
invalid_timestamp_count = int(label_analysis["_timestamp_dt"].isna().sum())
if invalid_timestamp_count:
    raise ValueError(f"Cannot define time-window labels with {invalid_timestamp_count} invalid timestamps.")

segment_keys_metrics = set(label_analysis["segment_id"].dropna())
segment_keys_report = set(segment_summary_input["segment_id"].dropna())
assert segment_keys_metrics == segment_keys_report, "Segment IDs differ between metrics and segment summary."
assert not label_analysis[list(required_identity)].isna().any().any(), "Sequence identity contains missing values."

print(f"Rows: {len(label_analysis):,}")
print(f"Machines: {label_analysis['machine_id'].nunique()}")
print(f"Runs: {label_analysis['run_id'].nunique()}")
print(f"Segments: {label_analysis['segment_id'].nunique()}")
print(f"Validated gap diagnostics loaded but not used as label inputs: {len(gap_validation_input)}")

### 3. Centralize every candidate threshold

**What the cell does:** Defines all rolling-window lengths, minimum observations, moderate/severe thresholds, context-relative thresholds, and balance-review limits in one configuration object.  
**Why it is important:** Central configuration makes assumptions visible and easy to challenge without searching through later code.  
**What to understand:** Moderate pressure uses trailing 30-second means; severe pressure uses trailing 60-second means. At least three observations are required, so one isolated row cannot trigger a signal.

In [ ]:
CONFIG = {
    "moderate_window_seconds": 30,
    "severe_window_seconds": 60,
    "minimum_observations": 3,
    "moderate": {
        "cpu_pct_mean": 80.0,
        "ram_pct_mean": 85.0,
        "swap_pct_mean": 60.0,
        "disk_latency_ms_mean": 5.0,
        "context_run_quantile": 0.95,
        "context_median_multiplier": 1.5,
    },
    "severe": {
        "cpu_pct_mean": 95.0,
        "ram_pct_mean": 95.0,
        "swap_pct_mean": 70.0,
        "disk_latency_ms_mean": 20.0,
        "context_run_quantile": 0.99,
        "context_median_multiplier": 2.0,
    },
    "too_few_positive_percentage": 1.0,
    "too_many_positive_percentage": 30.0,
    "concentration_warning_share": 0.80,
    "target_balance_percentage_for_ranking": 5.0,
}

print("Candidate threshold configuration:")
display(pd.json_normalize(CONFIG, sep="." ).T.rename(columns={0: "value"}))
print("Excluded from all rules: missed_deadline, sensor errors/timeouts, gap class, status, ended_at_utc, run_complete, temperature_c, gpu_usage_pct")

### 4. Calculate bounded sustained-resource windows

**What the cell does:** Stably sorts a copy and calculates trailing 30- and 60-second means separately inside each machine/run/segment. It also builds run-specific context-switch baselines.  
**Why it is important:** Time windows must reset at every segment, run, and machine boundary; otherwise a label could use observations across missing time or unrelated sequences.  
**What to understand:** Rolling values include the current row and recent history only, require at least three samples, and never cross a boundary.

In [ ]:
sequence_keys = ["machine_id", "run_id", "segment_id"]
label_analysis = label_analysis.sort_values(
    sequence_keys + ["_timestamp_dt", "_source_row"], kind="mergesort"
).copy()
rolling_metrics = ["cpu_pct", "ram_pct", "swap_pct", "disk_latency_ms", "context_switches_per_s"]

for window_name, seconds in [
    ("moderate", CONFIG["moderate_window_seconds"]),
    ("severe", CONFIG["severe_window_seconds"]),
]:
    for metric in rolling_metrics:
        label_analysis[f"{metric}_{window_name}_mean"] = np.nan
    for _, segment in label_analysis.groupby(sequence_keys, sort=False):
        ordered = segment.sort_values("_timestamp_dt")
        time_indexed = ordered.set_index("_timestamp_dt")
        for metric in rolling_metrics:
            rolled = time_indexed[metric].rolling(
                f"{seconds}s", min_periods=CONFIG["minimum_observations"]
            ).mean()
            label_analysis.loc[ordered.index, f"{metric}_{window_name}_mean"] = rolled.to_numpy()

context_baselines = (
    label_analysis.groupby(["machine_id", "run_id"])["context_switches_per_s"]
    .agg(
        context_run_median="median",
        context_run_p95=lambda values: values.quantile(CONFIG["moderate"]["context_run_quantile"]),
        context_run_p99=lambda values: values.quantile(CONFIG["severe"]["context_run_quantile"]),
    )
    .reset_index()
)
label_analysis = label_analysis.merge(context_baselines, on=["machine_id", "run_id"], how="left", validate="many_to_one")

print("Example bounded rolling values:")
display(label_analysis[sequence_keys + ["timestamp", "cpu_pct_moderate_mean", "cpu_pct_severe_mean"]].head(10))

### 5. Create moderate and severe resource-group signals

**What the cell does:** Converts sustained rolling means into five independent moderate signals and five severe signals.  
**Why it is important:** Resource groups must remain distinct; prohibited timing, timeout, outcome, temperature, and GPU fields cannot contribute to the label.  
**What to understand:** The displayed counts show how often each sustained resource condition occurs before candidate rules combine them.

In [ ]:
moderate = CONFIG["moderate"]
severe = CONFIG["severe"]

label_analysis["moderate_cpu"] = label_analysis["cpu_pct_moderate_mean"].ge(moderate["cpu_pct_mean"])
label_analysis["moderate_ram"] = label_analysis["ram_pct_moderate_mean"].ge(moderate["ram_pct_mean"])
label_analysis["moderate_swap"] = label_analysis["swap_pct_moderate_mean"].ge(moderate["swap_pct_mean"])
label_analysis["moderate_disk"] = label_analysis["disk_latency_ms_moderate_mean"].ge(moderate["disk_latency_ms_mean"])
label_analysis["moderate_context"] = (
    label_analysis["context_switches_per_s_moderate_mean"].ge(label_analysis["context_run_p95"])
    & label_analysis["context_switches_per_s_moderate_mean"].ge(
        moderate["context_median_multiplier"] * label_analysis["context_run_median"]
    )
)

label_analysis["severe_cpu"] = label_analysis["cpu_pct_severe_mean"].ge(severe["cpu_pct_mean"])
label_analysis["severe_ram"] = label_analysis["ram_pct_severe_mean"].ge(severe["ram_pct_mean"])
label_analysis["severe_swap"] = label_analysis["swap_pct_severe_mean"].ge(severe["swap_pct_mean"])
label_analysis["severe_disk"] = label_analysis["disk_latency_ms_severe_mean"].ge(severe["disk_latency_ms_mean"])
label_analysis["severe_context"] = (
    label_analysis["context_switches_per_s_severe_mean"].ge(label_analysis["context_run_p99"])
    & label_analysis["context_switches_per_s_severe_mean"].ge(
        severe["context_median_multiplier"] * label_analysis["context_run_median"]
    )
)

moderate_columns = ["moderate_cpu", "moderate_ram", "moderate_swap", "moderate_disk", "moderate_context"]
severe_columns = ["severe_cpu", "severe_ram", "severe_swap", "severe_disk", "severe_context"]
label_analysis["moderate_signal_count"] = label_analysis[moderate_columns].sum(axis=1).astype(int)
label_analysis["severe_signal_count"] = label_analysis[severe_columns].sum(axis=1).astype(int)

prohibited_label_inputs = {
    "missed_deadline", "sensor_errors_json", "revised_diagnostic_class", "status",
    "ended_at_utc", "run_complete", "temperature_c", "gpu_usage_pct"
}
actual_label_inputs = set(rolling_metrics)
assert not (prohibited_label_inputs & actual_label_inputs)

print("Sustained signal row counts:")
display(label_analysis[moderate_columns + severe_columns].sum().to_frame("row_count"))

### 6. Define the three candidate present-time rules

**What the cell does:** Tests Rule A (two moderate groups), Rule B (moderate CPU and RAM together), and Rule C (one severe group or two moderate groups).  
**Why it is important:** Comparing plausible definitions reveals sensitivity to rule design before any final label or future horizon is chosen.  
**What to understand:** These Boolean columns exist only in memory for analysis; Rule C can respond to one sustained severe signal, while high CPU alone cannot satisfy Rules A or B.

In [ ]:
RULES = {
    "rule_a_two_moderate_groups": {
        "definition": "At least two independent moderate 30-second resource signals.",
        "column": "rule_a_positive",
    },
    "rule_b_cpu_and_ram": {
        "definition": "Moderate 30-second CPU and RAM pressure occur together.",
        "column": "rule_b_positive",
    },
    "rule_c_severe_or_two_moderate": {
        "definition": "At least one severe 60-second resource signal or at least two moderate signals.",
        "column": "rule_c_positive",
    },
}

label_analysis["rule_a_positive"] = label_analysis["moderate_signal_count"].ge(2)
label_analysis["rule_b_positive"] = label_analysis["moderate_cpu"] & label_analysis["moderate_ram"]
label_analysis["rule_c_positive"] = (
    label_analysis["severe_signal_count"].ge(1)
    | label_analysis["moderate_signal_count"].ge(2)
)

print("Candidate positive counts:")
for rule_name, specification in RULES.items():
    positives = int(label_analysis[specification["column"]].sum())
    print(f"{rule_name}: {positives:,} — {specification['definition']}")
print("No future 5-minute or 10-minute target has been created.")

### 7. Measure continuous positive events

**What the cell does:** Counts consecutive positive periods and calculates their durations separately inside every machine/run/segment.  
**Why it is important:** Row counts alone can be misleading; one long episode and many short episodes have different meanings.  
**What to understand:** Event durations never cross a segment boundary, and a single isolated positive row has a duration of zero seconds.

In [ ]:
event_statistics = {}
event_tables = {}

for rule_name, specification in RULES.items():
    rule_column = specification["column"]
    events = []
    for sequence_key, sequence in label_analysis.groupby(sequence_keys, sort=False):
        ordered = sequence.sort_values("_timestamp_dt")
        positive = ordered[rule_column].fillna(False).astype(bool)
        event_start = positive & ~positive.shift(1, fill_value=False)
        event_number = event_start.cumsum()
        positive_rows = ordered.loc[positive].copy()
        positive_rows["_event_number"] = event_number.loc[positive].to_numpy()
        for event_id, event in positive_rows.groupby("_event_number"):
            start_time = event["_timestamp_dt"].min()
            end_time = event["_timestamp_dt"].max()
            events.append({
                "machine_id": sequence_key[0],
                "run_id": sequence_key[1],
                "segment_id": sequence_key[2],
                "event_number": int(event_id),
                "start_time": start_time,
                "end_time": end_time,
                "duration_seconds": float((end_time - start_time).total_seconds()),
                "row_count": int(len(event)),
            })
    event_table = pd.DataFrame(events)
    event_tables[rule_name] = event_table
    event_statistics[rule_name] = {
        "continuous_event_count": int(len(event_table)),
        "median_event_duration_seconds": float(event_table["duration_seconds"].median()) if len(event_table) else np.nan,
        "maximum_event_duration_seconds": float(event_table["duration_seconds"].max()) if len(event_table) else np.nan,
    }

display(pd.DataFrame(event_statistics).T)

### 8. Calculate machine and run distributions

**What the cell does:** Counts total and positive rows for every candidate rule by machine and by machine/run pair.  
**Why it is important:** A rule that labels only one machine or run may describe hardware identity or one experiment rather than general slowdown.  
**What to understand:** Percentages expose coverage and concentration differences across the three candidates.

In [ ]:
machine_distribution = (
    label_analysis.groupby("machine_id").size().rename("total_rows").reset_index()
)
run_distribution = (
    label_analysis.groupby(["machine_id", "run_id"]).size().rename("total_rows").reset_index()
)

for rule_name, specification in RULES.items():
    rule_column = specification["column"]
    machine_counts = label_analysis.groupby("machine_id")[rule_column].sum().astype(int)
    run_counts = label_analysis.groupby(["machine_id", "run_id"])[rule_column].sum().astype(int)
    machine_distribution[f"{rule_name}_positive_rows"] = machine_distribution["machine_id"].map(machine_counts).fillna(0).astype(int)
    machine_distribution[f"{rule_name}_positive_percentage"] = (
        machine_distribution[f"{rule_name}_positive_rows"] / machine_distribution["total_rows"] * 100
    ).round(4)
    run_distribution[f"{rule_name}_positive_rows"] = [
        int(run_counts.get((row.machine_id, row.run_id), 0)) for row in run_distribution.itertuples(index=False)
    ]
    run_distribution[f"{rule_name}_positive_percentage"] = (
        run_distribution[f"{rule_name}_positive_rows"] / run_distribution["total_rows"] * 100
    ).round(4)

print("Distribution by machine:")
display(machine_distribution)
print("Distribution by run:")
display(run_distribution)

### 9. Compare balance and recommend one provisional candidate

**What the cell does:** Combines positive rates, event statistics, coverage, maximum concentration, and explicit too-few/too-many checks. Eligible rules are ranked by lower concentration and then closeness to a 5% review target.  
**Why it is important:** A useful candidate needs enough examples and coverage across machines/runs, not merely a plausible formula.  
**What to understand:** `recommended_candidate=True` is a proposal for manual validation only; concentration warnings explain rejected or weaker candidates.

In [ ]:
comparison_records = []
total_rows = len(label_analysis)
total_machines = label_analysis["machine_id"].nunique()
total_runs = label_analysis["run_id"].nunique()

for rule_name, specification in RULES.items():
    rule_column = specification["column"]
    positive_rows = int(label_analysis[rule_column].sum())
    positive_percentage = positive_rows / total_rows * 100 if total_rows else 0.0
    machine_positive_counts = label_analysis.groupby("machine_id")[rule_column].sum()
    run_positive_counts = label_analysis.groupby("run_id")[rule_column].sum()
    positive_machine_count = int(machine_positive_counts.gt(0).sum())
    positive_run_count = int(run_positive_counts.gt(0).sum())
    max_machine_share = float(machine_positive_counts.max() / positive_rows) if positive_rows else np.nan
    max_run_share = float(run_positive_counts.max() / positive_rows) if positive_rows else np.nan
    too_few = positive_percentage < CONFIG["too_few_positive_percentage"]
    too_many = positive_percentage > CONFIG["too_many_positive_percentage"]
    only_one_machine = positive_machine_count <= 1
    only_one_run = positive_run_count <= 1
    concentration_warning = (
        (pd.notna(max_machine_share) and max_machine_share > CONFIG["concentration_warning_share"])
        or (pd.notna(max_run_share) and max_run_share > CONFIG["concentration_warning_share"])
    )
    if positive_rows == 0:
        assessment = "no_positive_cases"
    elif too_few:
        assessment = "too_few_positive_cases"
    elif too_many:
        assessment = "too_many_positive_cases"
    elif only_one_machine or only_one_run:
        assessment = "positives_limited_to_one_machine_or_run"
    elif concentration_warning:
        assessment = "positive_concentration_warning"
    else:
        assessment = "balanced_candidate_for_review"

    comparison_records.append({
        "rule_name": rule_name,
        "rule_definition": specification["definition"],
        "positive_rows": positive_rows,
        "positive_percentage": round(positive_percentage, 4),
        **event_statistics[rule_name],
        "positive_machine_count": positive_machine_count,
        "total_machine_count": int(total_machines),
        "positive_run_count": positive_run_count,
        "total_run_count": int(total_runs),
        "max_machine_share_of_positives": round(max_machine_share, 4) if pd.notna(max_machine_share) else np.nan,
        "max_run_share_of_positives": round(max_run_share, 4) if pd.notna(max_run_share) else np.nan,
        "positives_only_one_machine": only_one_machine,
        "positives_only_one_run": only_one_run,
        "too_few_positives": too_few,
        "too_many_positives": too_many,
        "balance_assessment": assessment,
    })

rule_comparison = pd.DataFrame(comparison_records)
eligible = rule_comparison.loc[rule_comparison["balance_assessment"].eq("balanced_candidate_for_review")].copy()
if len(eligible):
    eligible["_concentration_rank"] = eligible[["max_machine_share_of_positives", "max_run_share_of_positives"]].max(axis=1)
    eligible["_rate_distance"] = (eligible["positive_percentage"] - CONFIG["target_balance_percentage_for_ranking"]).abs()
    recommended_rule = eligible.sort_values(["_concentration_rank", "_rate_distance"]).iloc[0]["rule_name"]
else:
    recommended_rule = None

rule_comparison["recommended_candidate"] = rule_comparison["rule_name"].eq(recommended_rule)
rule_comparison["recommendation_status"] = np.where(
    rule_comparison["recommended_candidate"], "proposal_requires_manual_validation", "not_selected"
)

display(rule_comparison)

### 10. Save reports and verify input integrity

**What the cell does:** Saves the rule comparison and both distributions, validates them after reading, prints the final recommendation, and recomputes all protected-file checksums.  
**Why it is important:** The handoff must be reviewable and must prove that no source dataset, segment report, or gap report was changed.  
**What to understand:** The recommended rule remains provisional; the final checksum confirmation must be true and no future target exists.

In [ ]:
rule_comparison.to_csv(RULE_COMPARISON_PATH, index=False)
machine_distribution.to_csv(MACHINE_DISTRIBUTION_PATH, index=False)
run_distribution.to_csv(RUN_DISTRIBUTION_PATH, index=False)

saved_comparison = pd.read_csv(RULE_COMPARISON_PATH)
saved_machine_distribution = pd.read_csv(MACHINE_DISTRIBUTION_PATH)
saved_run_distribution = pd.read_csv(RUN_DISTRIBUTION_PATH)
assert len(saved_comparison) == len(RULES)
assert len(saved_machine_distribution) == total_machines
assert len(saved_run_distribution) == total_runs
assert saved_comparison["rule_name"].is_unique

hashes_after = {path: sha256_file(path) for path in protected_paths}
protected_inputs_unchanged = hashes_before == hashes_after

print("FINAL CANDIDATE-RULE COMPARISON")
display(rule_comparison[[
    "rule_name", "positive_rows", "positive_percentage", "continuous_event_count",
    "median_event_duration_seconds", "maximum_event_duration_seconds",
    "positive_machine_count", "positive_run_count", "balance_assessment",
    "recommended_candidate", "recommendation_status"
]])
print(f"Recommended candidate proposal: {recommended_rule}")
print("This recommendation requires manual validation and is not the final label rule.")
print("No future 5-minute or 10-minute target was created.")
print(f"All protected inputs remained unchanged: {protected_inputs_unchanged}")
print(f"Created: {RULE_COMPARISON_PATH}")
print(f"Created: {MACHINE_DISTRIBUTION_PATH}")
print(f"Created: {RUN_DISTRIBUTION_PATH}")

assert protected_inputs_unchanged, "A protected input changed during label analysis."
